# Workbook 02: Exploring word embeddings

В цій лабораторній ми детальніше розберемося з векторним представленням слів (word embeddings), а саме:

- Завантажимо вектори GloVe
- Перевіримо цікаві властивості цих векторів

У вас буде кілька завдань. Відповіді на завдання відправляють функції `lab.checkpoint()` та `lab.answer()`.

Для початку, заповніть параметр `email` нижче:

In [ ]:
!pip install --quiet --ignore-installed requests==2.32.4 https://static.lp-nlp.com/pypi/lpnlp-2025.12.14-py3-none-any.whl

In [ ]:
import lpnlp
lab = lpnlp.start(
    email="user.name@lpnu.ua",  # <---------------------- Заповніть це поле
    lab="exploring_word_embeddings"
    )


# GloVe

In [ ]:
!pip install gensim

Повний GloVe містить 4,000,000 векторів і займає багато пам'яті. Щоб уникнути проблем з пам'ятю, залишимо лише 50,000 векторів найчастотніших слів. Це трохи знизить якість моделей, але це зараз не головне.

In [ ]:
import gensim.downloader
from gensim.models import KeyedVectors
import tqdm


# Ця функція завантажить повний набір GloVe векторів та залишить `max_n` з них
def load_glove_subset(max_n: int) -> KeyedVectors:
    """Return top `max_n` word vectors. """
    all_glove = gensim.downloader.load("glove-wiki-gigaword-200")
    subset = KeyedVectors(all_glove.vector_size)
    for word in tqdm.tqdm(all_glove.key_to_index, total=max_n):
        if len(subset.vectors) >= max_n:
            break
        subset.add_vector(word, all_glove[word])
    return subset

# Цей виклик може зайняти хвилин 5-6, тож ми завантажимо вже готові 50k векторів
# glove = load_glove_subset(50_000)
glove = KeyedVectors.load("http://static.lp-nlp.com/files/glove-50k.bin")

In [ ]:
lab.checkpoint("glove subset size", len(glove.key_to_index))

Об'єкт `glove` може працювати як звичайний словник (`dict`), ключами якого є токени, а значеннями - вектори слів. Вектори, які ми завантажили, були натреновані на lower-cased текстах, тож і шукати вектори треба для слів у нижньому регістрі.

Подивимося, як виглядає вектор слова "ukraine":

In [ ]:
glove["ukraine"]

### Завдання

Знайдіть кількість та середнє значення (mean) елементів вектора слова "ukraine"

In [ ]:
##########################
# ВАШ КОД ТУТ:
size_v_ukraine = ...
mean_v_ukraine = ...
##########################
lab.checkpoint("size of 'ukraine'", size_v_ukraine)
lab.checkpoint("mean of 'ukraine'", mean_v_ukraine)

# Подібність слів

### Косинус подібності (cosine similarity)

Дуже часто метрикою близькості двох векторів слугує косинус подібності (cosine similarity). Косинус кута між двома векторами буде дорівнювати 1, коли напрями векторів точно збігаються. Для ортогональних векторів косинус дорівнює 0, а для протилежних -1.

Косинус кута можна порахувати через векторний добуток:

$$ \text{similarity}(p, q) = \cos {\theta} = \frac {p \cdot q} {|p| |q|} $$

$|x|$ означає довжину вектора $x$:

$$|x| = \sqrt {x ^ 2} $$

### Завдання

Напишіть функцію обчислення cosine similarity

In [ ]:
import numpy as np
from math import isclose

def normalize(v):
    return v / np.sqrt(v @ v)

def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    ###########################
    # Ваш код
    ###########################
    similarity = ...
    return similarity

# Знайдіть косинус подібності слів "apple" та "google"
cos_0 = cosine_similarity(glove["apple"], glove["google"])
assert isinstance(cos_0, float)
lab.checkpoint("cosine similarity between 'apple' and 'google'", float(cos_0))

### Завдання

Знайти подібність словa "awesome" до слів "amazing", great" та "bad"

In [ ]:
cos_1 = cosine_similarity(glove['awesome'], glove['amazing'])
lab.checkpoint("cosine_similarity between 'awesome' and 'amazing'", cos_1)

In [ ]:
cos_2 = ...
lab.checkpoint("cosine_similarity between 'awesome' and 'great'", cos_2)

In [ ]:
cos_3 = ...
lab.checkpoint("cosine_similarity between 'awesome' and 'bad'", cos_3)

## Пошук подібних слів

Завдяки векторному представленню, ми можемо уявляти собі слова точками в багатовимірному просторі. Близькі за семантикою (змістом) або морфологією (як одиниці мови) слова будуть відповідати близькими між собою точками.

Атож, найближчі сусіди слова будуть або синонимами, або антонімами, або виконувати ту саму роль в реченні.

Як знайти сусідів вектора певного слова? Наївне рішення — перебрати всі вектори і обчислити відстань до кожного з них. Для великих словників це може зайняти багато часу. Для великих словників або коли необхідно робити багато запитів, використовують методи приблизного пошуку векторів, такі як LSH або [faiss](https://github.com/facebookresearch/faiss). Ці методи попередньо будують індекси для векторів, які дозволяють здійснювати пошук подібних векторів дуже швидко (і з маленькою, але ненульовою ймовірністю пропускати деяких сусідів)

Оскільки наш словник є дуже невеликим (50 тисяч токенів), то можемо дозволити собі наївне рішення.

In [ ]:
from typing import List, Tuple

def find_similar(word: str , top_n: int = 10) -> List[Tuple[float, str]]:
    """Повертає `top_n` сусідів слова `word` разом з подібністю.

    Returns:
        list of (similarity, word) tuples.
    """
    # Рахуємо подібність вектора слова `word` з кожним іншим словом
    # Словник всіх слів міститься в `glove.vocab`

    ################### ВАШ КОД ТУТ
    result = ...
    ###############################

    return result


find_similar("awesome")

In [ ]:
find_similar("awful")

In [ ]:
find_similar("green")

In [ ]:
lab.checkpoint("most similar to 'green'", find_similar("green")[0][1])

## Швидший пошук сусідів (векторизований)

Наша функція `find_similar` є навмисно «наївною»: вона перебирає весь словник і обчислює косинусну подібність по одному слову за раз.

Ми можемо суттєво прискорити це, виконавши те саме обчислення за один крок:

- Нормалізуємо всі вектори слів один раз (тоді косинусна подібність зводиться до скалярного добутку).
- Для запитуваного слова беремо його нормалізований вектор і обчислюємо скалярний добуток з усією матрицею нормалізованих векторів.
- Вибираємо топ-N результатів за найвищими оцінками.

Це дозволяє уникнути циклів Python та використовує оптимізовану лінійну алгебру «під капотом».

In [ ]:
import numpy as np
from typing import List, Tuple

# Build a matrix of all vectors (shape: vocab_size x dim) and normalize it once.
all_words = glove.index_to_key
W = glove.vectors.astype(np.float32)

# Normalize each row to unit length
W_norm = W / np.linalg.norm(W, axis=1, keepdims=True)

def find_similar_fast(word: str, top_n: int = 10) -> List[Tuple[float, str]]:
    """Return the `top_n` nearest neighbors of `word` (fast vectorized version)."""

    if word not in glove.key_to_index:
        raise KeyError(f"Word '{word}' not in vocabulary")

    idx = glove.key_to_index[word]
    v = W_norm[idx]  # already normalized

    # Cosine similarity with all words (since both are normalized)
    sims = W_norm @ v

    # Exclude the query word itself
    sims[idx] = -np.inf

    # Get indices of the top_n highest similarities
    top_idx = np.argsort(-sims)[:top_n]

    return [(float(sims[i]), all_words[i]) for i in top_idx]

find_similar_fast("awesome")


Бібліотека `gensim`, за допомогою якої ми завантажували вектори, також вміє робити пошук подібних векторів, але значно швидше. Порівняємо з нашою імплементацією:

In [ ]:
glove.most_similar("facebook")

In [ ]:
find_similar("facebook")

### Завдання

Знайдіть найближчого сусіда до слова `cat` (одне слово)

In [ ]:
cat_top_synonym = ...
assert isinstance(cat_top_synonym, str)
lab.checkpoint("Most similar word to `cat`", cat_top_synonym)

### Завдання

Спробуйте функцію `find_similar` на різних словах. Які пари здалися вам найбільш цікавими, дивними або нелогічними?

In [ ]:
interesting_pair = ...    # пара слів, наприклад ("awesome", "aweful")
lab.checkpoint("Interesting/strange pairs", ...)

# Візуалізація

In [ ]:
from sklearn import manifold
import matplotlib.pyplot as plt


def plot_word_embeddings(model, search_list):
    words = []
    for term in search_list:
        words += [w[0] for w in model.most_similar([term], topn=5)]
    words += search_list
    words = list(set(words))

    vectors = model[words]

    tsne = manifold.TSNE(n_components=2, random_state=0, n_iter=10000, perplexity=7)
    T = tsne.fit_transform(vectors)

    plt.figure(figsize=(16, 10))
    plt.scatter(T[:, 0], T[:, 1])
    for label, x, y in zip(words, T[:, 0], T[:, 1]):
        plt.annotate(label, xy=(x+2, y+2), xytext=(0, 0), textcoords='offset points')

In [ ]:
plot_word_embeddings(glove, ["awesome", "terrible", "ukraine", "poland", "france", "red", "green", "cat", "two"])

# Векторна аріфметика

Ми можемо додавати та віднімати вектори слів. Результат таких операцій (який теж буде вектором) можна перетворити назад в слово. Для цього знайдемо кілька найближчих до результата векторів і подивимося, які слова їм відповідають.

Векторну аріфметику використовують для пошуку слів за аналогіями. Класичний приклад з першої статті про Word2Vec виглядає так:

```
V("king") - V("man") + V("woman") ~= V("queen")
```

In [ ]:
glove.similar_by_vector(glove["king"] - glove["man"] + glove["woman"])

### Завдання

Знайдіть найближчий вектор виразу
`paris` - `france` + `ukraine`

In [ ]:
answer = ...
lab.answer(answer)   # мені теж не подобається те, як написана відповідь :)

# Готово!

На наступному занятті ми:
- Використаємо ембедінги слів для класифікації текстів
- Дізнаємося, як влаштований word2vec
- Напишемо свою імплементацію word2vec
- Подивимося, як обійти проблему несловникових токенів
